In [18]:
import torch
from torch.utils.data import IterableDataset, DataLoader
import pandas as pd
import numpy as np
from pathlib import Path
import random
import pyarrow.parquet as pq


class FastDataset(IterableDataset):
    def __init__(self, folder_list):
        self.folder_list = folder_list
        self.union_feature_files = [
            ("feat1.parquet", [f'feat1_{j}' for j in range(10)]),
            ("feat2.parquet", [f'feat2_{j}' for j in range(10)]),
        ]
        self.separate_file_prefix = "large_data_part"
        self.separate_file_columns = [f'col_{k}' for k in range(100)]
        self.max_parts = 10
        self.dataset_len = 0
        for folder in self.folder_list:
            folder_path = Path(folder)
            parquet_file = pq.ParquetFile(folder_path / self.union_feature_files[0][0])
            total_rows = parquet_file.metadata.num_rows
            self.dataset_len += total_rows

    def reset(self):
        random.shuffle(self.folder_list)

    def __len__(self):
        return self.dataset_len

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        worker_id, num_workers = (worker_info.id, worker_info.num_workers) if worker_info else (0, 1)
        for folder in self.folder_list:
            folder_path = Path(folder)
            union_index = 0
            parquet_file = pq.ParquetFile(folder_path / self.union_feature_files[0][0])
            total_rows = parquet_file.metadata.num_rows
            if num_workers > 1:
                union_left = total_rows * worker_id // num_workers
                union_right = total_rows * (worker_id + 1) // num_workers
            else:
                union_left = 0
                union_right = total_rows

            for part in range(self.max_parts):
                separate_file = folder_path / f"{self.separate_file_prefix}{part}.parquet"
                if not separate_file.exists():
                    break
                parquet_file = pq.ParquetFile(separate_file)
                n_rows = parquet_file.metadata.num_rows
                union_index_new = union_index + n_rows
                if union_index_new <= union_left:
                    union_index = union_index_new
                    continue
                separate_df = parquet_file.read(columns=self.separate_file_columns).to_pandas()
                skipped_rows = max(0, union_left - union_index)
                omitted_rows = max(0, union_index_new - union_right)
                union_dfs = []
                
                for union_file, columns in self.union_feature_files:
                    union_file_path = folder_path / union_file
                    df = pd.read_parquet(union_file_path, columns=columns)
                    union_dfs.append(df.iloc[union_index + skipped_rows:union_index_new - omitted_rows].reset_index(drop=True))
                union_df = pd.concat(union_dfs, axis=1)
                df = pd.concat([separate_df.iloc[skipped_rows:-omitted_rows].reset_index(drop=True), union_df], axis=1)
                union_index = union_index_new
                for _, row in df.iterrows():
                    yield torch.tensor(row.values, dtype=torch.float32)
                if union_index >= union_right:
                    break
            
            assert union_index == union_right, f"union_index: {union_index}, union_right: {union_right}"

In [20]:
import datetime
import pandas as pd 
import numpy as np 
from pathlib import Path
import time

start_date = datetime.date(2025, 1, 1)
end_date = datetime.date(2025, 1, 31)

folder_list = []
for i in range((end_date - start_date).days + 1):
    current_date = start_date + datetime.timedelta(days=i)
    folder = Path(f"example/data_{current_date}")
    folder_list.append(str(folder))

dataset = FastDataset(folder_list)
print(f"Dataset length: {len(dataset)}")
# shuffling is achieved by shuffling folder_list
dataloader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=False, prefetch_factor=2, persistent_workers=False, multiprocessing_context="spawn")
print(f"len(dataloader): {len(dataloader)} batches")

for epoch in range(5):
    t1 = time.time()
    dataset.reset()
    t2 = time.time()
    print(f"Epoch {epoch}, reset time: {t2 - t1:.6f} seconds")
    for batch_idx, batch in enumerate(dataloader):
        pass
    t3 = time.time()
    print(f"Epoch {epoch}, data loading time: {t3 - t2:.6f} seconds")


Dataset length: 310000
len(dataloader): 4844 batches
Epoch 0, reset time: 0.000023 seconds


Traceback (most recent call last):
Traceback (most recent call last):
  File "<string>", line 1, in <module>
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Users/runtianzhai/mambaforge/envs/ml/lib/python3.11/multiprocessing/spawn.py", line 122, in spawn_main
Traceback (most recent call last):
  File "/Users/runtianzhai/mambaforge/envs/ml/lib/python3.11/multiprocessing/spawn.py", line 122, in spawn_main
  File "<string>", line 1, in <module>
  File "<string>", line 1, in <module>
  File "/Users/runtianzhai/mambaforge/envs/ml/lib/python3.11/multiprocessing/spawn.py", line 122, in spawn_main
  File "/Users/runtianzhai/mambaforge/envs/ml/lib/python3.11/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
    exitcode = _main(fd, parent_sentinel)
    exitcode = _main(fd, parent_sentinel)
    exitcode = _main(fd, parent_sentinel)  
                      ^ ^ ^ ^  ^ ^ ^  ^   ^  ^ ^ ^       ^ ^ ^       ^  ^ ^^^^ ^^^^^^^

RuntimeError: DataLoader worker (pid(s) 59619) exited unexpectedly

# Build example dataset

In [34]:
import datetime
import pandas as pd 
import numpy as np 
from pathlib import Path

start_date = datetime.date(2025, 1, 1)
end_date = datetime.date(2025, 3, 31)
sid_single = 9995 // 5 
sid_all = 2050

for i in range((end_date - start_date).days + 1):
    current_date = start_date + datetime.timedelta(days=i)
    # Create a dataframe with 10,000 rows and 10 columns of random values
    folder = Path(f"example/data_{current_date}")
    folder.mkdir(parents=True, exist_ok=True)
    df1 = pd.DataFrame(np.random.rand(9995, 10), columns=[f'feat1_{j}' for j in range(10)])
    df1.to_parquet(f'example/data_{current_date}/feat1.parquet')
    df2 = pd.DataFrame(np.random.rand(9995, 10), columns=[f'feat2_{j}' for j in range(10)])
    df2.to_parquet(f'example/data_{current_date}/feat2.parquet')
    sid_this = np.random.choice(sid_all, sid_single, replace=False)
    sid_this = np.tile(sid_this, 5)
    df_meta = pd.DataFrame({'sid': sid_this}, index=df2.index)
    df_meta.to_parquet(f'example/data_{current_date}/meta.parquet')
    df_target = pd.DataFrame(np.random.rand(9995, 1), columns=['target'])
    df_target.to_parquet(f'example/data_{current_date}/target.parquet')
    

In [32]:
for i in range((end_date - start_date).days + 1):
    current_date = start_date + datetime.timedelta(days=i)
    sizes = [sid_single] * 5
    for j in range(5):
        size = sizes[j]
        df = pd.DataFrame(np.random.rand(size, 100), columns=[f'col_{k}' for k in range(100)])
        df.to_parquet(f'example/data_{current_date}/large_data_part{j}.parquet')

In [35]:
df = pd.read_parquet("/Users/runtianzhai/Documents/project/Python/kaggle/example/data_2025-01-01/meta.parquet")
df

,sid
0,560
1,1971
2,1643
3,1799
4,823
...,...
9990,1093
9991,1051
9992,1046
9993,1402


In [37]:
df = pd.read_parquet("/Users/runtianzhai/Documents/project/Python/kaggle/example_560.parquet")
df

,sid,feat1_0,feat1_1,feat1_2,feat1_3,feat1_4,feat1_5,feat1_6,feat1_7,feat1_8,...,col_91,col_92,col_93,col_94,col_95,col_96,col_97,col_98,col_99,folder
0,560,0.749610,0.880962,0.909237,0.436903,0.491864,0.889539,0.210044,0.948659,0.137700,...,0.626352,0.786129,0.591862,0.389491,0.632150,0.140905,0.433474,0.858639,0.015397,example/data_2025-01-01
1,560,0.876123,0.874630,0.284907,0.128560,0.554481,0.198198,0.496624,0.429724,0.488607,...,0.379800,0.326610,0.233373,0.696677,0.492459,0.779308,0.043484,0.459323,0.360980,example/data_2025-01-01
2,560,0.039121,0.451196,0.824210,0.750675,0.417903,0.764763,0.462646,0.070681,0.216975,...,0.023330,0.669322,0.235576,0.816427,0.236155,0.033013,0.826244,0.028233,0.529361,example/data_2025-01-01
3,560,0.317771,0.502515,0.061075,0.723137,0.067233,0.637894,0.828492,0.564187,0.809500,...,0.495890,0.444863,0.818900,0.309935,0.335675,0.508352,0.034013,0.433613,0.710995,example/data_2025-01-01
4,560,0.938745,0.140069,0.620075,0.725161,0.503005,0.646795,0.471708,0.763434,0.998658,...,0.402832,0.540810,0.570284,0.582789,0.693498,0.690987,0.307536,0.353313,0.529216,example/data_2025-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
440,560,0.797394,0.395035,0.955859,0.407654,0.870593,0.525596,0.384343,0.918910,0.593225,...,0.269412,0.568136,0.614690,0.859200,0.371703,0.037751,0.107811,0.414955,0.959591,example/data_2025-03-31
441,560,0.956512,0.307147,0.945456,0.649926,0.924227,0.543123,0.292500,0.902086,0.746355,...,0.561733,0.416726,0.686545,0.948729,0.794051,0.026616,0.379500,0.729911,0.707591,example/data_2025-03-31
442,560,0.031361,0.004714,0.813411,0.183038,0.781907,0.862304,0.227491,0.689089,0.471690,...,0.000551,0.651245,0.308851,0.674450,0.058799,0.706857,0.474662,0.832119,0.969353,example/data_2025-03-31
443,560,0.365769,0.405146,0.594624,0.870683,0.569609,0.907410,0.916556,0.557975,0.593862,...,0.365156,0.754028,0.999403,0.877541,0.294530,0.393739,0.606936,0.272270,0.061286,example/data_2025-03-31


In [39]:
df["target"]

0      0.578043
1      0.944809
2      0.070774
3      0.050348
4      0.492073
         ...   
440    0.091374
441    0.336691
442    0.984795
443    0.266503
444    0.136686
Name: target, Length: 445, dtype: float64

In [41]:
df = pd.read_parquet("/Users/runtianzhai/Documents/project/Python/kaggle/temp/all_sids.parquet")
df

,sid,count
0,0,440
1,1,445
2,2,440
3,3,445
4,4,440
...,...,...
2045,2045,435
2046,2046,445
2047,2047,430
2048,2048,435


In [42]:
df["count"].describe()

count    2050.000000
mean      438.804878
std         7.412149
min       405.000000
25%       435.000000
50%       440.000000
75%       445.000000
max       450.000000
Name: count, dtype: float64

In [43]:
df["sid"].unique()

array([   0,    1,    2, ..., 2047, 2048, 2049], shape=(2050,))

In [44]:
df_pred = pd.read_parquet("/Users/runtianzhai/Documents/project/Python/kaggle/temp/pred/pred_sid_0.parquet")
df_pred

,target,pred
0,0.891745,0.890563
1,0.044240,0.044844
2,0.315886,0.315869
3,0.760109,0.759823
4,0.018018,0.019513
...,...,...
435,0.714568,0.713891
436,0.578992,0.578886
437,0.622994,0.623095
438,0.642396,0.642818
